In [38]:
import zipfile
from pathlib import Path
import shutil


# ============================================================
# CONFIGURAÇÃO
# ============================================================

pasta_origem = Path(r"dados\raw\cliente\zip")

pasta_staging = Path(
    r"dados\raw\cliente\csv"
)


# ============================================================
# CRIAR DIRETÓRIO DE SAÍDA
# ============================================================

pasta_staging.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CONTADORES
# ============================================================

total_zip = 0
total_csv = 0
total_erros = 0


# ============================================================
# CONTROLE
# ============================================================

arquivos_extraidos = []
arquivos_erro = []


# ============================================================
# 1. LOCALIZAR TODOS OS ZIP
# ============================================================

print("=" * 70)
print("ETAPA 1 - LOCALIZAÇÃO DOS ARQUIVOS ZIP")
print("=" * 70)

arquivos_zip = list(
    pasta_origem.rglob("*.zip")
)

total_zip = len(
    arquivos_zip
)

print(
    f"Total de arquivos ZIP encontrados: {total_zip}"
)

for arquivo_zip in arquivos_zip:
    print(
        f"ZIP | {arquivo_zip}"
    )


# ============================================================
# 2. EXTRAIR SOMENTE OS CSV
# ============================================================

print("\n")
print("=" * 70)
print("ETAPA 2 - EXTRAÇÃO DOS CSV")
print("=" * 70)


for arquivo_zip in arquivos_zip:

    try:

        # ----------------------------------------------------
        # NOME DA PASTA ONDE O ZIP ESTÁ
        #
        # Exemplo:
        # dados\raw\cliente\BRASIL\relatorio1.zip
        #
        # Resultado:
        # BRASIL
        # ----------------------------------------------------

        nome_pasta = (
            arquivo_zip.parent.name
        )


        # ----------------------------------------------------
        # NOME DO ZIP SEM EXTENSÃO
        #
        # relatorio1.zip
        # →
        # relatorio1
        # ----------------------------------------------------

        nome_zip = (
            arquivo_zip.stem
        )


        print("\n")
        print(
            f"PROCESSANDO: {arquivo_zip}"
        )


        # ----------------------------------------------------
        # ABRIR ZIP
        # ----------------------------------------------------

        with zipfile.ZipFile(
            arquivo_zip,
            "r"
        ) as zip_ref:


            # ------------------------------------------------
            # LISTAR ARQUIVOS INTERNOS
            # ------------------------------------------------

            arquivos_internos = (
                zip_ref.namelist()
            )


            # ------------------------------------------------
            # LOCALIZAR SOMENTE CSV
            # ------------------------------------------------

            arquivos_csv = [
                arquivo
                for arquivo in arquivos_internos
                if arquivo.lower().endswith(".csv")
            ]


            print(
                f"CSV encontrados: "
                f"{len(arquivos_csv)}"
            )


            # ------------------------------------------------
            # EXTRAIR CADA CSV
            # ------------------------------------------------

            for nome_csv in arquivos_csv:

                try:

                    # ----------------------------------------
                    # NOME ORIGINAL DO CSV
                    # ----------------------------------------

                    nome_csv_original = (
                        Path(
                            nome_csv
                        ).name
                    )


                    # ----------------------------------------
                    # NOME DO CSV SEM EXTENSÃO
                    # ----------------------------------------

                    nome_csv_base = (
                        Path(
                            nome_csv
                        ).stem
                    )


                    # ----------------------------------------
                    # NOME FINAL
                    #
                    # BRASIL_relatorio1_dados.csv
                    # ----------------------------------------

                    nome_saida = (
                        f"{nome_pasta}_"
                        f"{nome_zip}_"
                        f"{nome_csv_base}.csv"
                    )


                    arquivo_saida = (
                        pasta_staging /
                        nome_saida
                    )


                    # ----------------------------------------
                    # EXTRAIR CSV DIRETAMENTE DO ZIP
                    # ----------------------------------------

                    with zip_ref.open(
                        nome_csv
                    ) as origem:

                        with open(
                            arquivo_saida,
                            "wb"
                        ) as destino:

                            shutil.copyfileobj(
                                origem,
                                destino
                            )


                    # ----------------------------------------
                    # CONTADOR
                    # ----------------------------------------

                    total_csv += 1


                    # ----------------------------------------
                    # CONTROLE
                    # ----------------------------------------

                    arquivos_extraidos.append({

                        "pasta": nome_pasta,

                        "zip": arquivo_zip.name,

                        "csv_original":
                            nome_csv_original,

                        "arquivo_saida":
                            nome_saida

                    })


                    # ----------------------------------------
                    # LOG
                    # ----------------------------------------

                    print(
                        f"   OK | "
                        f"{nome_csv_original} → "
                        f"{nome_saida}"
                    )


                except Exception as e:

                    total_erros += 1

                    arquivos_erro.append({

                        "tipo": "CSV",

                        "zip": str(
                            arquivo_zip
                        ),

                        "csv": nome_csv,

                        "erro": str(e)

                    })


                    print(
                        f"   ERRO CSV | "
                        f"{nome_csv} | "
                        f"{e}"
                    )


    except Exception as e:

        total_erros += 1

        arquivos_erro.append({

            "tipo": "ZIP",

            "zip": str(
                arquivo_zip
            ),

            "csv": "",

            "erro": str(e)

        })


        print(
            f"ERRO ZIP | "
            f"{arquivo_zip} | "
            f"{e}"
        )


# ============================================================
# 3. RELATÓRIO FINAL
# ============================================================

print("\n")
print("=" * 70)
print("RELATÓRIO FINAL")
print("=" * 70)

print(
    f"ZIP encontrados : {total_zip}"
)

print(
    f"CSV extraídos   : {total_csv}"
)

print(
    f"Erros           : {total_erros}"
)

print(
    f"Pasta de saída  : {pasta_staging}"
)


# ============================================================
# 4. LISTAR ARQUIVOS EXTRAÍDOS
# ============================================================

print("\n")
print("=" * 70)
print("ARQUIVOS CSV EXTRAÍDOS")
print("=" * 70)


arquivos_saida = list(
    pasta_staging.glob("*.csv")
)


for arquivo in arquivos_saida:

    print(
        f"CSV | {arquivo.name}"
    )


print(
    f"\nTotal de CSV na pasta: "
    f"{len(arquivos_saida)}"
)


# ============================================================
# 5. ERROS
# ============================================================

if arquivos_erro:

    print("\n")
    print("=" * 70)
    print("ERROS")
    print("=" * 70)

    for erro in arquivos_erro:

        print(
            f"Tipo: {erro['tipo']}\n"
            f"ZIP: {erro['zip']}\n"
            f"CSV: {erro['csv']}\n"
            f"Erro: {erro['erro']}\n"
        )

else:

    print("\nNenhum erro encontrado.")

ETAPA 1 - LOCALIZAÇÃO DOS ARQUIVOS ZIP
Total de arquivos ZIP encontrados: 4
ZIP | dados\raw\cliente\zip\CLIENTE1\relatorio1.zip
ZIP | dados\raw\cliente\zip\CLIENTE1\relatorio2.zip
ZIP | dados\raw\cliente\zip\CLIENTE2\relatorio1.zip
ZIP | dados\raw\cliente\zip\CLIENTE2\relatorio2.zip


ETAPA 2 - EXTRAÇÃO DOS CSV


PROCESSANDO: dados\raw\cliente\zip\CLIENTE1\relatorio1.zip
CSV encontrados: 1
   OK | dados.csv → CLIENTE1_relatorio1_dados.csv


PROCESSANDO: dados\raw\cliente\zip\CLIENTE1\relatorio2.zip
CSV encontrados: 1
   OK | dados.csv → CLIENTE1_relatorio2_dados.csv


PROCESSANDO: dados\raw\cliente\zip\CLIENTE2\relatorio1.zip
CSV encontrados: 1
   OK | dados.csv → CLIENTE2_relatorio1_dados.csv


PROCESSANDO: dados\raw\cliente\zip\CLIENTE2\relatorio2.zip
CSV encontrados: 1
   OK | dados.csv → CLIENTE2_relatorio2_dados.csv


RELATÓRIO FINAL
ZIP encontrados : 4
CSV extraídos   : 4
Erros           : 0
Pasta de saída  : dados\raw\cliente\csv


ARQUIVOS CSV EXTRAÍDOS
CSV | CLIENTE1_relatorio